# pyLSTemp v1.7 - exemplos completos de uso

Este notebook demonstra a API pública da versão `1.6.0` usando matrizes sintéticas pequenas. Em um fluxo real, substitua essas matrizes por bandas Landsat lidas com `rasterio`, `xarray` ou outra ferramenta geoespacial.

Funções demonstradas:

- `spectral_index(...)`
- `brightness(...)`
- `emissivity(...)`
- `water_vapor(...)`
- `single_window(...)`
- `split_window(...)`
- `list_algorithms(...)`

## 1. Importar bibliotecas e funções públicas

In [ ]:
import numpy as np

from pylstemp import (
    spectral_index,
    brightness,
    emissivity,
    water_vapor,
    single_window,
    split_window,
    list_algorithms,
)
import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 2. Criar dados de exemplo

Aqui usamos arrays pequenos apenas para demonstrar a chamada das funções. Em dados reais:

- `red`: banda vermelha.
- `nir`: banda do infravermelho próximo.
- `blue`: banda azul.
- `band_10`: banda termal 10 em DN.
- `band_11`: banda termal 11 em DN.

Os valores de `rad_gain` e `rad_bias` podem vir do metadado da cena. Na versão atual, se não forem informados, a biblioteca usa os valores padrão cadastrados para o sensor.

In [ ]:
blue = np.array([
    [0.08, 0.09, 0.10, 0.11, 0.12],
    [0.08, 0.09, 0.10, 0.11, 0.12],
    [0.09, 0.10, 0.11, 0.12, 0.13],
    [0.10, 0.11, 0.12, 0.13, 0.14],
    [0.11, 0.12, 0.13, 0.14, 0.15],
], dtype=float)

red = np.array([
    [0.18, 0.20, 0.24, 0.28, 0.30],
    [0.19, 0.22, 0.25, 0.29, 0.31],
    [0.20, 0.23, 0.27, 0.32, 0.35],
    [0.21, 0.24, 0.30, 0.36, 0.40],
    [0.22, 0.26, 0.33, 0.39, 0.45],
], dtype=float)

nir = np.array([
    [0.46, 0.50, 0.55, 0.58, 0.60],
    [0.48, 0.53, 0.57, 0.61, 0.64],
    [0.50, 0.56, 0.62, 0.66, 0.70],
    [0.52, 0.60, 0.68, 0.72, 0.76],
    [0.55, 0.64, 0.72, 0.78, 0.82],
], dtype=float)

band_10 = np.array([
    [1000, 1010, 1020, 1030, 1040],
    [1010, 1020, 1030, 1040, 1050],
    [1020, 1030, 1040, 1050, 1060],
    [1030, 1040, 1050, 1060, 1070],
    [1040, 1050, 1060, 1070, 1080],
], dtype=float)

band_11 = np.array([
    [900, 910, 920, 930, 940],
    [910, 920, 930, 940, 950],
    [920, 930, 940, 950, 960],
    [930, 940, 950, 960, 970],
    [940, 950, 960, 970, 980],
], dtype=float)

sensor = "landsat_8"

print("Shape:", band_10.shape)

## 3. `spectral_index(...)`: calcular NDVI e EVI

A forma padronizada para calcular índices espectrais é usar `spectral_index(index=...)`. O NDVI usa `nir` e `red`; o EVI usa `nir`, `red` e `blue`. No artigo usado como referência, a equação do EVI aparece nomeada como SARVI2: `2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))`.

In [ ]:
ndvi_image = spectral_index(index="ndvi", nir=nir, red=red)
evi_image = spectral_index(index="evi", nir=nir, red=red, blue=blue)

print("NDVI:")
print(np.round(ndvi_image, 4))
print("\nEVI:")
print(np.round(evi_image, 4))

## 4. `brightness(...)`: calcular temperatura de brilho

A função `brightness(...)` calcula a temperatura de brilho da banda termal selecionada.

Use:

- `band="band_10"` para a banda termal 10.
- `band="band_11"` para a banda termal 11.

`rad_gain` corresponde a `RADIANCE_MULT_BAND_X` e `rad_bias` corresponde a `RADIANCE_ADD_BAND_X`. Se omitidos, a biblioteca usa os valores padrão do sensor.

In [ ]:
brightness_10 = brightness(
    band_10,
    band="band_10",
    sensor=sensor,
)

brightness_11 = brightness(
    band_11,
    band="band_11",
    sensor=sensor,
)

print("Brightness band 10:")
print(np.round(brightness_10, 2))
print("\nBrightness band 11:")
print(np.round(brightness_11, 2))

### 4.1. Sobrescrever `rad_gain` e `rad_bias` manualmente

Use esta forma quando quiser informar diretamente os valores do metadado da cena.

In [ ]:
brightness_10_custom = brightness(
    band_10,
    band="band_10",
    sensor=sensor,
    rad_gain=0.0003342,
    rad_bias=0.1,
)

print(np.round(brightness_10_custom, 2))

## 5. `emissivity(...)`: calcular emissividade

A função `emissivity(...)` recebe o NDVI já calculado e retorna a emissividade para o fluxo da banda selecionada.

Use:

- `band="band_10"` para emissividade associada ao fluxo da banda 10.
- `band="band_11"` para emissividade associada ao fluxo da banda 11.

O método padrão é `avdan-2016`, recomendado para fluxo single-channel. Para split-window, prefira métodos com emissividade específica por banda, como `gopinadh-2018` ou `xiaolei-2014`.

In [ ]:
emissivity_10 = emissivity(
    ndvi_image,
    band="band_10",
    band_4_red=red,
    emissivity_method="avdan-2016",
)

emissivity_11 = emissivity(
    ndvi_image,
    band="band_11",
    band_4_red=red,
    emissivity_method="gopinadh-2018",
)

print("Emissivity band 10:")
print(np.round(emissivity_10, 4))
print("\nEmissivity band 11:")
print(np.round(emissivity_11, 4))

## 6. `single_window(...)`: calcular LST por single-channel

O fluxo single-channel usa a temperatura de brilho da banda 10 já calculada. A função `single_window(...)` calcula internamente o NDVI e a emissividade a partir das bandas vermelho/NIR informadas.

In [ ]:
lst_single_kelvin = single_window(
    brightness_band_10=brightness_10,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="mono-window-2016",
    emissivity_method="avdan-2016",
    unit="kelvin",
)

lst_single_celsius = single_window(
    brightness_band_10=brightness_10,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="mono-window-2016",
    emissivity_method="avdan-2016",
    unit="celsius",
)

print("Single-window LST Kelvin:")
print(np.round(lst_single_kelvin, 2))
print("\nSingle-window LST Celsius:")
print(np.round(lst_single_celsius, 2))

## 7. `water_vapor(...)`: estimar vapor d'água

A função `water_vapor(...)` estima vapor d'água precipitável com o método selecionado.

Atualmente:

- `method="wang-2015"`

`window_size=5` representa uma janela local de `5 x 5` pixels. `group_count=5` controla quantos grupos locais baseados em NDVI são usados no método. Para dados pequenos como este exemplo, usamos `group_count=1` para evitar grupos vazios.

Observação: como este notebook usa dados sintéticos muito pequenos, criamos abaixo uma versão auxiliar da banda 11 com relação local bem comportada para demonstrar uma saída finita do método Wang 2015. Em dados reais, use a temperatura de brilho real da banda 11.

In [ ]:
brightness_11_for_water = np.nanmean(brightness_10) + 0.8 * (brightness_10 - np.nanmean(brightness_10))

water = water_vapor(
    brightness_band_10=brightness_10,
    brightness_band_11=brightness_11_for_water,
    ndvi_image=ndvi_image,
    method="wang-2015",
    window_size=5,
    group_count=1,
)

print("Water vapor:")
print(np.round(water, 4))

## 8. `split_window(...)`: calcular LST por split-window

O fluxo split-window usa as temperaturas de brilho das bandas 10 e 11 já calculadas. O método `du-2015` pode usar `water_vapor`; se `water_vapor=None`, usa o conjunto geral de coeficientes de Du et al. para `[0.0, 6.3] g/cm2`.

In [ ]:
lst_split_du = split_window(
    brightness_band_10=brightness_10,
    brightness_band_11=brightness_11,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="du-2015",
    emissivity_method="gopinadh-2018",
    unit="celsius",
)

print("Split-window Du 2015 LST Celsius:")
print(np.round(lst_split_du, 2))

### 8.1. Split-window com vapor d'água pixel a pixel

O método `jimenez-munoz-2014` exige `water_vapor`. O raster estimado por `water_vapor(method="wang-2015")` pode ser passado diretamente.

In [ ]:
lst_split_jimenez = split_window(
    brightness_band_10=brightness_10,
    brightness_band_11=brightness_11,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="jimenez-munoz-2014",
    emissivity_method="gopinadh-2018",
    water_vapor=water,
    unit="celsius",
)

print("Split-window Jimenez-Munoz 2014 LST Celsius:")
print(np.round(lst_split_jimenez, 2))

## 9. `list_algorithms(...)`: listar famílias e métodos disponíveis

In [ ]:
catalog = list_algorithms()

for family, methods in catalog.items():
    if family == "credit":
        continue
    print(f"{family}: {list(methods.keys())}")

## 10. Resumo do fluxo típico

1. Calcule `ndvi_image` com `spectral_index(index="ndvi", nir=nir, red=red)`.
2. Calcule `brightness_10` e, se necessário, `brightness_11` com `brightness(...)`.
3. Para single-channel, use `single_window(...)` com `brightness_10`.
4. Para split-window, use `split_window(...)` com `brightness_10` e `brightness_11`.
5. Se o método exigir vapor d'água, estime com `water_vapor(...)` ou informe um valor/raster externo.